In [1]:
import pandas as pd
import fitz
import cv2
from io import BytesIO
from PIL import Image
import base64
from openai import OpenAI
from pydantic import BaseModel
import json
import numpy as np
import traceback
from typing import Optional
import io

import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Alignment, Border, Side, Font

from blank_functions.forms.form_recognition import FormRecognition
from blank_functions.ui.ui_functions import get_pic_from_pdf, save_to_excel, get_correct_answers, postprocess_raw_output, check_answers, final_styling, extract_text_from_image, transform_json_to_dataframe
from blank_functions.ui.ui_functions import promt, prepare_cur_dict, reorder_cols

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# no limit columns for pandas
pd.set_option('display.max_columns', None)

c:\Users\zamko\Documents\mom_project\repo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# def get_row_image(cur_form, answer):
#     row_images = list()
#     for cell, i in zip(answer.cells, range(1, 11)):
#         if i <=5:
#             x, y, w, h = cell.x, cell.y, cell.w, cell.h
#             cell_image = cur_form.raw_image[y:y+h, x:x+w]
#             row_images.append(cell_image)

#     # объединить изображения в одно
#     row_image = np.concatenate(row_images, axis=1)
#     row_image = cv2.resize(row_image, (64, 16), interpolation=cv2.INTER_AREA)
#     return row_image

# def set_row_images(form):
#     for row in ["user_id", "version"]:
#         getattr(form, row).row_image = get_row_image(form, getattr(form, row))
#     for j in range(1, 11):
#         answer_row = getattr(form, f"answer{j}")
#         correction_row = getattr(form, f"correction{j}")
#         answer_row.row_image = get_row_image(form, answer_row)
#         correction_row.row_image = get_row_image(form, correction_row)
#         if correction_row.cells[0].user_value != None:
#             answer_row.row_image = correction_row.row_image.copy()



# def save_to_excel(df_global_styled, form_dict):
#     with pd.ExcelWriter('output_with_images.xlsx', engine='xlsxwriter') as writer:
#         # Сохраняем сам DataFrame на лист (например, Sheet1)
#         df_global_styled.to_excel(writer, sheet_name='Sheet1', index=False)
        
#         # Получаем объект worksheet, чтобы работать с картинками
#         workbook  = writer.book
#         worksheet = writer.sheets['Sheet1']


#         # Теперь итерируемся по строкам DF
#         for row_idx in range(len(df_global_styled)):
#             form = form_dict[row_idx]
#             excel_row = row_idx + 1
            
#             excel_col = df_global_styled.columns.get_loc(f'Картинка код участника')
#             img_data = io.BytesIO()
#             img = Image.fromarray(getattr(form, 'user_id').row_image)
#             img.save(img_data, format='PNG')
#             img_data.seek(0)  # переходим в начало буфера
            
#             worksheet.insert_image(
#                 excel_row, 
#                 excel_col,
#                 "some_name.png", 
#                 {'image_data': img_data}
#             )

#             excel_col = df_global_styled.columns.get_loc(f'Картинка вариант')
#             img_data = io.BytesIO()
#             img = Image.fromarray(getattr(form, 'version').row_image)
#             img.save(img_data, format='PNG')
#             img_data.seek(0)  # переходим в начало буфера

#             worksheet.insert_image(
#                 excel_row, 
#                 excel_col,
#                 "some_name.png", 
#                 {'image_data': img_data}
#             )
            
#             for i in range(1, 11):
#                 excel_col = df_global_styled.columns.get_loc(f'Картинка ответа {i}')
#                 img_data = io.BytesIO()
#                 img = Image.fromarray(getattr(form, f'answer{i}').row_image)
#                 img.save(img_data, format='PNG')
#                 img_data.seek(0)  # переходим в начало буфера
                
#                 worksheet.insert_image(
#                     excel_row, 
#                     excel_col,
#                     "some_name.png", 
#                     {'image_data': img_data}
#                 )

# def reorder_cols(df_global_styled):
#     df_global_styled[f'Картинка код участника'] = ''
#     df_global_styled[f'Картинка вариант'] = ''
    
#     for i in range(1, 11):
#         df_global_styled[f'Картинка ответа {i}'] = ''


#     reorder_cols_list = ['Предмет', 'Код участника', 'Картинка код участника', 'Вариант', 'Картинка вариант']
#     for i in range(1, 11):  
#         reorder_cols_list.append(f'Задание {i}')
#         reorder_cols_list.append(f'Картинка ответа {i}')

#     for i in range(1, 11):
#         reorder_cols_list.append(f'Начисленные баллы {i}')

#     reorder_cols_list.append('Начисленные баллы сумма')

#     df_global_styled = df_global_styled[reorder_cols_list]
#     return df_global_styled


In [3]:
import time

import os
# /Users/vladislav/Documents/mom_project_repo/streamlit-exam-form-app/notebooks/docling_test.ipynb
# os.chdir("C:\\Users\\zamko\\Documents\\mom_project\\repo\\notebooks")
# pdf_path = "./data/valid_format/valid_questions.pdf"
# answers_path = "./data/answers.xlsx"
# template_path = "./data/template_2.jpg"
# json_path = "./data/rows_data.json"

pdf_path = "C:/Users/zamko/Documents/mom_project/repo/data/valid_format/valid_questions.pdf"
answers_path = "C:/Users/zamko/Documents/mom_project/repo/data/answers.xlsx"
template_path = "C:/Users/zamko/Documents/mom_project/repo/data/template_2.jpg"
json_path = "C:/Users/zamko/Documents/mom_project/repo/data/rows_data.json"



pdf_bytes = open(pdf_path, 'rb').read()
pdf_document = fitz.open(stream=pdf_bytes, filetype="pdf")
num_pages = pdf_document.page_count

answers_bytes = open(answers_path, 'rb').read()
answers = pd.read_excel(BytesIO(answers_bytes))

cur_version = 2

df_global = pd.DataFrame()
form_dict = {}
for i in range(0, 1):
    cur_pic = get_pic_from_pdf(pdf_bytes, i, zoom=6.0)
    form = FormRecognition(
        image = cur_pic,
        template_path = template_path,
        json_path = json_path,
        answers = answers,
        version = cur_version)
    form = form.run_pipeline()
    cur_dict = prepare_cur_dict(form)
    df_current = transform_json_to_dataframe(cur_dict)
    df_global = pd.concat([df_global, df_current]).reset_index(drop=True)
    form_dict[i] = form

correct_answers = get_correct_answers(answers_bytes)
df_global_processed = postprocess_raw_output(df_global, correct_answers)
df_global_answers = check_answers(df_global_processed)
df_global_styled = final_styling(df_global_answers)
df_global_styled = reorder_cols(df_global_styled)
save_to_excel(df_global_styled, form_dict)


C:\Users\zamko\Documents\mom_project\repo\blank_functions\ui\ui_functions.py:177: FutureWarning: Passing bytes to 'read_excel' is deprecated and will be removed in a future version. To read from a byte string, wrap it in a `BytesIO` object.
  correct_answers = pd.read_excel(correct_answers_path)
